# Olist E-Commerce: Bronze Layer
**Dataset**: [Brazilian E-commerce Public Dataset by Olist](https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce)       
**Layer**: Bronze: Raw ingestion from source files  
 **Tools Used**: Databricks, PySpark, Delta Lake  

## Overview
This notebook ingests all 9 raw CSV files from the Olist Brazilian e-commerce dataset into Delta tables, forming the bronze layer of a three-tier Medallion Architecture pipeline.

This bronze layer is a faithful, unmodified copy of the source data. No cleaning, transformation, or filtering is applied. Only an `ingestion_timestamp` audit column is added to each table to record when the data was loaded. All data quality issues identified in the source files are addressed in the silver layer.

## Source Files
| Table | File | Rows |
|---|---|---|
| `bronze_customers` | olist_customers_dataset.csv | 99,441 |
| `bronze_orders` | olist_orders_dataset.csv | 99,441 |
| `bronze_order_items` | olist_order_items_dataset.csv | 112,650 |
| `bronze_order_payments` | olist_order_payments_dataset.csv | 103,886 |
| `bronze_order_reviews` | olist_order_reviews_dataset.csv | 104,162 |
| `bronze_products` | olist_products_dataset.csv | 32,951 |
| `bronze_sellers` | olist_sellers_dataset.csv | 3,095 |
| `bronze_geolocation` | olist_geolocation_dataset.csv | 1,000,163 |
| `bronze_product_category_translation` | product_category_name_translation.csv | 71 |

## Design Decisions
- **`inferSchema=True`** is used for brevity. A production implementation would define explicit schemas per table to ensure type stability across runs.
- **Overwrite mode** ensures that re-running the notebook 
  reloads the source data cleanly without duplication.
- **All 9 tables are ingested in a single loop** driven by a filename dictionary, making it straightforward to add new source files.

In [0]:
from pyspark.sql.functions import current_timestamp

base_path = "/Volumes/workspace/default/olist_raw/"
# Path to raw CSV files in Databricks Unity Catalog Volume.

tables = {
    "customers": "olist_customers_dataset.csv",
    "orders": "olist_orders_dataset.csv",
    "order_items": "olist_order_items_dataset.csv",
    "order_payments": "olist_order_payments_dataset.csv",
    "order_reviews": "olist_order_reviews_dataset.csv",
    "products": "olist_products_dataset.csv",
    "sellers": "olist_sellers_dataset.csv",
    "geolocation": "olist_geolocation_dataset.csv",
    "product_category_translation": "product_category_name_translation.csv"
}
for table_name, file_name in tables.items():
    df = (
        spark.read.format("csv")
        .option("header", True)
        .option("inferSchema", True)
        .load(base_path + file_name)
    )
    
    # Add ingestion metadata
    df = df.withColumn("ingestion_timestamp", current_timestamp())
    
    # Write as Delta table
    df.write.format("delta") \
        .mode("overwrite") \
        .saveAsTable(f"bronze_{table_name}")
    
    print(f"Loaded {table_name}")

Loaded customers
Loaded orders
Loaded order_items
Loaded order_payments
Loaded order_reviews
Loaded products
Loaded sellers
Loaded geolocation
Loaded product_category_translation


In [0]:
# Verifies that all 9 tables were created successfully and reports row counts.A FAILED status with an error message indicates the table was not written. Row counts should be stable across runs since the source files are static.
from pyspark.sql.types import StructType, StructField, StringType, LongType

tables = [
    "bronze_customers",
    "bronze_orders",
    "bronze_order_items",
    "bronze_order_payments",
    "bronze_order_reviews",
    "bronze_products",
    "bronze_sellers",
    "bronze_geolocation",
    "bronze_product_category_translation"
]

results = []

for table in tables:
    try:
        df = spark.table(table)
        row_count = df.count()
        results.append((table, "SUCCESS", row_count, None))

    except Exception as e:
        results.append((table, "FAILED", None, str(e)))

schema = StructType([
    StructField("table", StringType(), False),
    StructField("status", StringType(), False),
    StructField("row_count", LongType(), True),
    StructField("error_message", StringType(), True)
])

display(spark.createDataFrame(results, schema))

table,status,row_count,error_message
bronze_customers,SUCCESS,99441,null
bronze_orders,SUCCESS,99441,null
bronze_order_items,SUCCESS,112650,null
bronze_order_payments,SUCCESS,103886,null
bronze_order_reviews,SUCCESS,104162,null
bronze_products,SUCCESS,32951,null
bronze_sellers,SUCCESS,3095,null
bronze_geolocation,SUCCESS,1000163,null
bronze_product_category_translation,SUCCESS,71,null


In [0]:
for table in tables:
    print(f"\n{'='*60}")
    print(f"Schema: {table}")
    print('='*60)
    spark.table(table).printSchema()


Schema: bronze_customers
root
 |-- customer_id: string (nullable = true)
 |-- customer_unique_id: string (nullable = true)
 |-- customer_zip_code_prefix: integer (nullable = true)
 |-- customer_city: string (nullable = true)
 |-- customer_state: string (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = true)


Schema: bronze_orders
root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = true)


Schema: bronze_order_items
root
 |-- order_id: string (nullable = true)
 |-- order_item_id: integer (nullable = true)
 |-- product_id: string (nullable = true)
